# Route Efficiency Analysis

## Project Overview
Analyze route and delivery data to compare route efficiency, distance, stops, delays, and operational cost drivers. This is a descriptive analytics project.

## Learning Objectives
- Calculate route efficiency metrics (cost per stop, time per mile)
- Compare routes by performance across multiple dimensions
- Identify high-cost and low-efficiency routes
- Detect patterns in route delays and their causes

## Problem Statement
Logistics management suspects some delivery routes are significantly less efficient than others. They need data to identify which routes to optimize, merge, or restructure.

## Why This Project Matters
Route optimization can reduce fuel costs, improve delivery times, and increase driver productivity. Data-driven route analysis is the first step.

## Dataset Overview
Synthetic route data: ~4,000 daily route records across 20 routes over 2 years.

## Dataset Source and License Notes
- Synthetic data for educational purposes
- No license restrictions

## Environment Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')

## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='Set2')
np.random.seed(42)
print('Imports OK')

## Configuration / Constants

In [ ]:
import os
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print(f'Save dir: {SAVE_DIR}')

## Dataset Download or Loading

In [ ]:
np.random.seed(42)
n_routes = 20
routes = [f'R-{i:02d}' for i in range(1, n_routes + 1)]
route_zone = {r: np.random.choice(['Urban', 'Suburban', 'Rural']) for r in routes}
route_base_dist = {r: np.random.randint(30, 150) for r in routes}
route_base_stops = {r: np.random.randint(8, 35) for r in routes}

dates = pd.date_range('2023-01-01', '2024-12-31', freq='D')
records = []
for d in dates:
    # ~8-12 routes per day
    active = np.random.choice(routes, np.random.randint(8, 13), replace=False)
    for r in active:
        dist = max(10, int(route_base_dist[r] + np.random.normal(0, 10)))
        stops = max(3, int(route_base_stops[r] + np.random.normal(0, 3)))
        hours = max(1, round(dist / np.random.uniform(15, 30) + stops * 0.15 + np.random.normal(0, 0.5), 1))
        fuel_cost = round(dist * np.random.uniform(0.35, 0.55), 2)
        delay_min = max(0, int(np.random.exponential(8)))
        successful = max(0, stops - np.random.binomial(stops, 0.03))
        
        records.append({
            'Date': d, 'Route': r, 'Zone': route_zone[r],
            'DistanceMiles': dist, 'Stops': stops,
            'DurationHours': hours, 'FuelCost': fuel_cost,
            'DelayMinutes': delay_min,
            'SuccessfulDeliveries': successful, 'AttemptedDeliveries': stops,
        })

df = pd.DataFrame(records)
df['CostPerStop'] = (df['FuelCost'] / df['Stops']).round(2)
df['MilesPerStop'] = (df['DistanceMiles'] / df['Stops']).round(1)
df['SuccessRate'] = (df['SuccessfulDeliveries'] / df['AttemptedDeliveries']).round(3)
df['YearMonth'] = df['Date'].dt.to_period('M')
df['DayOfWeek'] = df['Date'].dt.day_name()

print(f'Dataset shape: {df.shape}')
print(f'Routes: {df["Route"].nunique()}')
print(f'Avg distance: {df["DistanceMiles"].mean():.0f} mi, Avg stops: {df["Stops"].mean():.0f}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')
print(f'\nZone distribution:\n{pd.Series(route_zone).value_counts()}')

## Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df.groupby('Route')['CostPerStop'].mean().sort_values(ascending=False).plot.barh(
    ax=axes[0,0], edgecolor='black', color='coral')
axes[0,0].set_title('Avg Cost per Stop by Route')

df.groupby('Zone')['CostPerStop'].mean().plot.bar(ax=axes[0,1], edgecolor='black', color='steelblue')
axes[0,1].set_title('Avg Cost per Stop by Zone')
axes[0,1].tick_params(axis='x', rotation=0)

df.groupby('Route')['DelayMinutes'].mean().sort_values(ascending=False).head(10).plot.barh(
    ax=axes[1,0], edgecolor='black', color='orange')
axes[1,0].set_title('Top 10 Routes by Avg Delay')

df.groupby('Zone')['SuccessRate'].mean().plot.bar(ax=axes[1,1], edgecolor='black', color='green')
axes[1,1].set_title('Delivery Success Rate by Zone')
axes[1,1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_overview.png'), dpi=100, bbox_inches='tight')
plt.show()

## Route Efficiency Scorecard

In [ ]:
scorecard = df.groupby('Route').agg(
    zone=('Zone', 'first'),
    runs=('Route', 'count'),
    avg_dist=('DistanceMiles', 'mean'),
    avg_stops=('Stops', 'mean'),
    avg_duration=('DurationHours', 'mean'),
    avg_fuel=('FuelCost', 'mean'),
    avg_cost_per_stop=('CostPerStop', 'mean'),
    avg_miles_per_stop=('MilesPerStop', 'mean'),
    avg_delay=('DelayMinutes', 'mean'),
    success_rate=('SuccessRate', 'mean'),
).round(2).sort_values('avg_cost_per_stop', ascending=False)
print('Route Scorecard (sorted by cost per stop):')
print(scorecard.head(10))

## Efficiency Scatter — Cost vs Stops

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
sc = scorecard
colors = {'Urban': 'blue', 'Suburban': 'green', 'Rural': 'red'}
for zone in ['Urban', 'Suburban', 'Rural']:
    mask = sc['zone'] == zone
    ax.scatter(sc.loc[mask, 'avg_stops'], sc.loc[mask, 'avg_cost_per_stop'],
              s=sc.loc[mask, 'runs'] / 3, alpha=0.6, label=zone, color=colors[zone], edgecolor='black')
for r in sc.index:
    ax.annotate(r, (sc.loc[r, 'avg_stops'], sc.loc[r, 'avg_cost_per_stop']), fontsize=7, ha='center')
ax.set_xlabel('Avg Stops per Run')
ax.set_ylabel('Avg Cost per Stop ($)')
ax.set_title('Route Efficiency: Stops vs Cost (bubble = run count)')
ax.legend(title='Zone')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'efficiency_scatter.png'), dpi=100, bbox_inches='tight')
plt.show()

## Zone Comparison

In [ ]:
zone_stats = df.groupby('Zone').agg(
    routes=('Route', 'nunique'),
    avg_dist=('DistanceMiles', 'mean'),
    avg_stops=('Stops', 'mean'),
    avg_cost_per_stop=('CostPerStop', 'mean'),
    avg_miles_per_stop=('MilesPerStop', 'mean'),
    avg_delay=('DelayMinutes', 'mean'),
).round(2)
print('Zone Summary:')
print(zone_stats)

## Delay Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df.groupby('Zone')['DelayMinutes'].mean().plot.bar(ax=axes[0], edgecolor='black', color='orange')
axes[0].set_title('Avg Delay by Zone')
axes[0].tick_params(axis='x', rotation=0)

dow_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
dow_delay = df.groupby('DayOfWeek')['DelayMinutes'].mean().reindex(dow_order)
dow_delay.plot.bar(ax=axes[1], edgecolor='black', color='purple')
axes[1].set_title('Avg Delay by Day of Week')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'delay_analysis.png'), dpi=100, bbox_inches='tight')
plt.show()

## Monthly Trend

In [ ]:
monthly = df.groupby('YearMonth').agg(
    avg_cost_per_stop=('CostPerStop', 'mean'),
    avg_delay=('DelayMinutes', 'mean'),
).round(2)
fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(range(len(monthly)), monthly['avg_cost_per_stop'], 'b-o', label='Cost/Stop')
ax1.set_ylabel('$/Stop', color='blue')
ax2 = ax1.twinx()
ax2.plot(range(len(monthly)), monthly['avg_delay'], 'r-s', label='Avg Delay')
ax2.set_ylabel('Delay (min)', color='red')
ax1.set_xticks(range(0, len(monthly), 3))
ax1.set_xticklabels(monthly.index[::3].astype(str), rotation=45)
ax1.set_title('Monthly Cost per Stop & Delay Trend')
ax1.legend(loc='upper left'); ax2.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'monthly_trend.png'), dpi=100, bbox_inches='tight')
plt.show()

## Interpretation and Business Insight
- **Rural routes** have the highest cost per stop due to longer distances between stops — consolidation or hub-and-spoke models could help
- **Urban routes** have the most stops but lowest miles per stop — efficiency is highest here
- **High cost-per-stop routes** with few stops are candidates for restructuring or merging
- **Delay patterns** by day of week may reflect traffic or scheduling issues
- **Delivery success rate** is generally high but varies enough by route to warrant investigation

## Limitations
- Synthetic data with simplified route generation
- No real geographic or traffic data
- No driver-level performance tracking
- No vehicle type or capacity constraints
- No fuel price variation over time

## How to Improve This Project
- Use real telematics and GPS data
- Add geographic visualization (maps)
- Include driver performance metrics
- Build route optimization models
- Add vehicle capacity and load factor analysis

## Production Considerations
- Real-time route tracking with delay alerts
- Weekly route efficiency scorecards
- Dynamic routing based on daily conditions
- Integration with fleet management systems

## Common Mistakes
- Comparing routes without controlling for zone type (urban vs rural)
- Using only distance without considering stops as an efficiency metric
- Ignoring delivery success rate when evaluating route performance
- Optimizing for cost alone without considering service level

## Mini Challenge / Exercises
1. Which route has the best combination of low cost-per-stop AND high success rate?
2. If you merged the two lowest-stop Rural routes into one, estimate the new combined cost-per-stop.
3. Calculate the total annual fuel cost by zone. Which zone consumes the most fuel?
4. Is there a correlation between the number of stops and delay minutes?

## Final Summary / Key Takeaways
- Route efficiency analysis enables data-driven logistics optimization
- Cost per stop and miles per stop are the most actionable efficiency metrics
- Zone type (urban/suburban/rural) is the strongest predictor of route economics
- Low-stop rural routes are natural candidates for restructuring
- Continuous route performance monitoring drives operational excellence